# Portugol lexer and parser (notebook version)

Earlier notebook version of the Portugol interpreter built with SLY: a lexer for the Portuguese-keyword teaching language and a parser that evaluates declarations, assignments, `se`/`senao`, `para` loops, `imprima` and `leia`.

In [ ]:
#   https://sly.readthedocs.io/en/latest/sly.html
from sly import Lexer, Parser

In [ ]:
class CalcLexer(Lexer):
    tokens = {INSTANCIA, FIMACAO,ID, NUM, PLUS, TIMES, MINUS, DIVIDE, EQ, LPAREN, RPAREN,\
              BEGIN,INTEGER,PRINT,READ,IF,THEN,ELSE,FIMSE,END,FOR,TO,STEP,FIMPARA,LT,GT,STRING}
    
    ignore = '\t'

    # Tokens
    STRING = r'\".*\"'
    ID = r'[a-zA-Z_][a-zA-Z0-9_]*'
    NUM = r'\d+'
    PLUS = r'\+'
    MINUS = r'-'
    TIMES = r'\*'
    DIVIDE = r'/'
    EQ = r'='
    LPAREN = r'\('
    RPAREN = r'\)'
    LT = r'\<'
    GT = r'\>'
    FIMACAO = r'\;'
    INSTANCIA = r'\:'


    # Special cases (reserved words of Portugol, kept in Portuguese on purpose)
    ID['inicio']   = BEGIN
    ID['inteiro']  = INTEGER
    ID['print_stmt']  = PRINT
    ID['read_stmt']     = READ
    ID['if_stmt']       = IF
    ID['entao']    = THEN
    ID['senao']    = ELSE
    ID['fim_se']   = FIMSE
    ID['fim']      = END
    ID['for_stmt']     = FOR
    ID['ate']      = TO
    ID['passo']    = STEP
    ID['fim_para'] = FIMPARA

    # Ignored pattern
    ignore_newline = r'\n+'

    # Extra action for newlines
    def ignore_newline(self, t):
        self.lineno += t.value.count('\n')
        
    # Ignored pattern
    ignore_space = r' '

    # Extra action for newlines
    def ignore_space(self, t):
        self.lineno += t.value.count(' ')

    def error(self, t):
        print("Illegal character '%s'" % t.value[0])
        self.index += 1

In [ ]:
class CalcParser(Parser):
    tokens = CalcLexer.tokens
    
    precedence = (
        ('left',PLUS,MINUS),
        ('left',TIMES,DIVIDE)
    )
    
    def __init__(self):
        self.vars = {}
        print('\nInitializing parser... \n')
        
#---------------------------------------------------------------        
        
    @_("BEGIN commands END")
    def initialD(self,p):
        print(p.commands)
        return p.commands    
    
#---------------------------------------------------------------

    @_("command commands")
    def commands(self,p):
        return str(p.command) + "\n" + p.commands    
        
    @_("command")
    def commands(self,p):
        return str(p.command)    
    
#---------------------------------------------------------------        

    @_("instantiation")
    def command(self,p):
        return p.instantiation
    @_("assignment")
    def command(self,p):
        return p.assignment
    @_("if_command")
    def command(self,p):
        return p.if_command
    @_("for_command")
    def command(self,p):
        return p.for_command
    @_("print_stmt")
    def command(self,p):
        return p.print_stmt
    @_("read_stmt")
    def command(self,p):
        return p.read_stmt       
    
#---------------------------------------------------------------
    
    @_("INTEGER ID FIMACAO")
    def instantiation(self,p):
        self.vars[p.ID] = '0'
        return p.ID + " = 0"
    @_("INTEGER ID INSTANCIA int_expr FIMACAO")
    def instantiation(self,p):
        self.vars[p.ID] = p.int_expr
        return p.ID + " = " + str(p.int_expr)
    
#---------------------------------------------------------------

    @_("ID INSTANCIA int_expr FIMACAO")
    def assignment(self,p):
        self.vars[p.ID] = p.int_expr
        return p.ID + " = " + str(p.int_expr)  

#---------------------------------------------------------------

    @_("IF comparison THEN commands FIMSE")
    def if_command(self,p):        
        if p.comparison:
            return p.commands
        else:
            pass
        return p.comparison
    
    @_("IF comparison THEN commands ELSE commands FIMSE")
    def if_command(self,p):
        if p.comparison:
            return p.comandos0
        else:
            return p.comandos1
        
    #---------------------------------------------------------------

    @_("int_expr EQ int_expr")
    def comparison(self,p):
        return p.int_expr0 == p.int_expr1
    
    @_("int_expr GT int_expr")
    def comparison(self,p):
        return p.int_expr0 > p.int_expr1
    
    @_("int_expr LT int_expr")
    def comparison(self,p):
        return p.int_expr0 < p.int_expr1

#---------------------------------------------------------------        
    
    @_("FOR int_expr TO int_expr STEP int_expr commands FIMPARA")
    def for_command(self,p):
        string = ""
        for x in range(p.int_expr0,p.int_expr1,p.int_expr2):
            string += p.commands + "\n"
        
        return string
        #return p.int_expr0  
    
#---------------------------------------------------------------        
    
    @_("PRINT LPAREN STRING RPAREN FIMACAO")
    def print_stmt(self,p):
        #print(p.STRING)
        return p.STRING  
    @_("PRINT LPAREN int_expr RPAREN FIMACAO")
    def print_stmt(self,p):
        #print(p.int_expr)
        return p.int_expr 
     
#---------------------------------------------------------------        
    
    @_("READ LPAREN ID RPAREN FIMACAO")
    def read_stmt(self,p):
        a = input("Input (numeric): ")
        self.vars[p.ID] = int(a)
        print()
        return "read_stmt -> "+p.ID +" = "+ a  

#---------------------------------------------------------------
    
    @_("ID")
    def int_expr(self,p):
        try:
            return self.vars[p.ID]  
        except:
            return p.ID
    @_("NUM")
    def int_expr(self,p):
        return int(p.NUM)
    @_("int_expr PLUS int_expr")
    def int_expr(self,p):
        return p.expr0 + p.expr1        
    @_("int_expr MINUS int_expr")
    def int_expr(self,p):
        return p.expr0 - p.expr1        
    @_("int_expr TIMES int_expr")
    def int_expr(self,p):
        return p.expr0 * p.expr1        
    @_("int_expr DIVIDE int_expr")
    def int_expr(self,p):
        return p.expr0 / p.expr1


In [ ]:
path = 'example.portugol'
fp = open(path,"r")
text = fp.read()
fp.close

lexer = CalcLexer()

for tok in lexer.tokenize(text):
        print('token=%r, lexeme=%r' % (tok.type, tok.value))

        
parser = CalcParser()

result = parser.parse(lexer.tokenize(text))
if result == None:
    print('The code does not belong to the language!')
else:
    print('Code is correct!')